In [1]:
!pip install rank_bm25 -q

import pandas as pd
import numpy as np
import pickle
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm

df = pd.read_csv('/kaggle/input/datasets/b22dckh131hongkhnhvn/arxiv-preprocessed/arxiv_preprocessed.csv')
df = df.dropna(subset=['text_tfidf']).reset_index(drop=True)
print(f"Số paper: {len(df):,}")

Số paper: 100,000


## Vai trò của notebook

Notebook này đảm nhiệm phần **keyword retrieval** trong hệ thống truy xuất paper:

- **TF-IDF + Cosine Similarity**: tìm theo mức độ trùng khớp từ khóa có trọng số.
- **BM25**: baseline truy xuất thông tin mạnh hơn TF-IDF trong nhiều bài toán search.
- **Boolean Search**: hỗ trợ lọc AND/OR/NOT, phù hợp khi người dùng cần điều kiện rõ ràng.

Phần bổ sung bên dưới sửa Boolean ranking và thêm đánh giá định lượng để kết quả không chỉ dừng ở demo thủ công.


In [2]:
print("Đang build TF-IDF matrix...")

tfidf_vectorizer = TfidfVectorizer(
    max_features = 100_000,   # giữ 100k từ phổ biến nhất
    ngram_range  = (1, 2),   # unigram + bigram: "deep learning", "neural network"
    min_df       = 3,        # bỏ từ xuất hiện < 3 paper
    max_df       = 0.85,     # bỏ từ xuất hiện > 85% paper (quá phổ biến)
    sublinear_tf = True,     # dùng log(tf) thay vì tf thô → chuẩn hơn
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['text_tfidf'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]:,} papers × {tfidf_matrix.shape[1]:,} terms")
print(f"  → Sparse density: {tfidf_matrix.nnz / (tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.4%}")

Đang build TF-IDF matrix...
TF-IDF matrix shape: (100000, 100000)
  → 100,000 papers × 100,000 terms
  → Sparse density: 0.1165%


In [3]:
print("Đang build BM25 index (mất ~2-3 phút)...")

# BM25 cần list of list of tokens
corpus_tokens = [text.split() for text in tqdm(df['text_tfidf'])]
bm25 = BM25Okapi(
    corpus_tokens,
    k1 = 1.5,    # điều chỉnh ảnh hưởng của term frequency
    b  = 0.75,   # điều chỉnh ảnh hưởng của độ dài document
)

print(f"BM25 index built! Corpus size: {len(corpus_tokens):,} docs")

Đang build BM25 index (mất ~2-3 phút)...


  0%|          | 0/100000 [00:00<?, ?it/s]

BM25 index built! Corpus size: 100,000 docs


In [4]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk
nltk.download('stopwords', quiet=True)

STOP_WORDS  = set(stopwords.words('english'))
stemmer     = PorterStemmer()

def preprocess_query(query: str) -> str:
    """Xử lý query giống pipeline TV1"""
    query  = query.lower()
    query  = re.sub(r'[^a-z\s]', ' ', query)
    tokens = query.split()
    tokens = [stemmer.stem(t) for t in tokens
              if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

# ─────────────────────────────────────────────
def search_tfidf(query: str, top_k: int = 10) -> pd.DataFrame:
    q_processed = preprocess_query(query)
    q_vec       = tfidf_vectorizer.transform([q_processed])
    scores      = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx     = scores.argsort()[::-1][:top_k]

    results = df.iloc[top_idx][['id','title','categories','year']].copy()
    results['score']  = scores[top_idx].round(4)
    results['method'] = 'TF-IDF'
    return results.reset_index(drop=True)

# ─────────────────────────────────────────────
def search_bm25(query: str, top_k: int = 10) -> pd.DataFrame:
    q_processed = preprocess_query(query)
    q_tokens    = q_processed.split()
    scores      = bm25.get_scores(q_tokens)
    top_idx     = scores.argsort()[::-1][:top_k]

    results = df.iloc[top_idx][['id','title','categories','year']].copy()
    results['score']  = scores[top_idx].round(4)
    results['method'] = 'BM25'
    return results.reset_index(drop=True)

In [5]:
def compare(query: str, top_k: int = 5):
    print(f"\n{'='*65}")
    print(f"  Query: \"{query}\"")
    print(f"  Processed: \"{preprocess_query(query)}\"")
    print('='*65)

    r_tfidf = search_tfidf(query, top_k)
    r_bm25  = search_bm25(query,  top_k)

    print(f"\n{'─'*65}")
    print(f"  TF-IDF (top {top_k})")
    print(f"{'─'*65}")
    for i, row in r_tfidf.iterrows():
        print(f"  {i+1}. [{row['score']:.4f}] {row['title'][:70]}")
        print(f"       {row['categories']}  ({row['year']})")

    print(f"\n{'─'*65}")
    print(f"  BM25 (top {top_k})")
    print(f"{'─'*65}")
    for i, row in r_bm25.iterrows():
        print(f"  {i+1}. [{row['score']:.4f}] {row['title'][:70]}")
        print(f"       {row['categories']}  ({row['year']})")

# Chạy thử nhiều loại query
compare("transformer attention mechanism natural language processing")
compare("image classification convolutional neural network")
compare("reinforcement learning reward optimization")
compare("graph neural network node classification")


  Query: "transformer attention mechanism natural language processing"
  Processed: "transform attent mechan natur languag process"

─────────────────────────────────────────────────────────────────
  TF-IDF (top 5)
─────────────────────────────────────────────────────────────────
  1. [0.2853] Transformer Dissection: A Unified Understanding of Transformer's   Att
       cs.LG stat.ML  (2019)
  2. [0.2678] TensorCoder: Dimension-Wise Attention via Tensor Representation for   
       cs.CL cs.LG  (2020)
  3. [0.2645] Attention Boosted Sequential Inference Model
       cs.CL  (2018)
  4. [0.2610] An Empirical Study of Spatial Attention Mechanisms in Deep Networks
       cs.CV cs.CL cs.LG  (2019)
  5. [0.2340] Language models and Automated Essay Scoring
       cs.CL cs.LG stat.ML  (2019)

─────────────────────────────────────────────────────────────────
  BM25 (top 5)
─────────────────────────────────────────────────────────────────
  1. [21.7508] A Tensorized Transformer for Language Mo

In [6]:
# Build một lần để Boolean Search không phải split toàn bộ corpus mỗi lần gọi.
doc_token_sets = df['text_tfidf'].fillna('').str.split().apply(set).to_list()
title_token_sets = df['title'].fillna('').apply(preprocess_query).str.split().apply(set).to_list()

inverted_index = {}
for doc_id, words in enumerate(doc_token_sets):
    for word in words:
        inverted_index.setdefault(word, set()).add(doc_id)


def search_boolean(query: str, mode: str = 'AND', top_k: int = 10,
                   rank_method: str = 'BM25') -> pd.DataFrame:
    """
    Boolean Retrieval dùng inverted index.

    AND: paper phải chứa tất cả token trong query.
    OR : paper chứa ít nhất một token.
    NOT: paper chứa token đầu tiên và không chứa token thứ hai.

    Sau khi lọc bằng Boolean, kết quả được rank lại bằng BM25/TF-IDF để tránh
    trường hợp nhiều paper có cùng match_score nhưng paper mới hơn lại đứng trên.
    """
    tokens = preprocess_query(query).split()
    if not tokens:
        return pd.DataFrame(columns=['id', 'title', 'categories', 'year', 'match_score', 'rank_score', 'method'])

    postings = [inverted_index.get(token, set()) for token in tokens]

    if mode == 'AND':
        candidate_ids = set.intersection(*postings) if postings else set()
    elif mode == 'OR':
        candidate_ids = set.union(*postings) if postings else set()
    elif mode == 'NOT':
        include = postings[0] if postings else set()
        exclude = postings[1] if len(postings) >= 2 else set()
        candidate_ids = include - exclude
    else:
        raise ValueError("mode phải là 'AND', 'OR' hoặc 'NOT'")

    if not candidate_ids:
        return pd.DataFrame(columns=['id', 'title', 'categories', 'year', 'match_score', 'rank_score', 'method'])

    candidate_ids = np.array(sorted(candidate_ids))

    if rank_method.upper() == 'BM25':
        scores = bm25.get_scores(tokens)
        rank_scores = scores[candidate_ids]
    elif rank_method.upper() == 'TF-IDF':
        q_vec = tfidf_vectorizer.transform([' '.join(tokens)])
        rank_scores = cosine_similarity(q_vec, tfidf_matrix[candidate_ids]).flatten()
    else:
        raise ValueError("rank_method phải là 'BM25' hoặc 'TF-IDF'")

    results = df.iloc[candidate_ids][['id', 'title', 'categories', 'year']].copy()
    results['match_score'] = [
        sum(token in doc_token_sets[i] for token in tokens)
        for i in candidate_ids
    ]
    results['title_match_score'] = [
        sum(token in title_token_sets[i] for token in tokens)
        for i in candidate_ids
    ]
    results['rank_score'] = rank_scores.round(4)
    results['method'] = f'Boolean-{mode}+{rank_method.upper()}'

    return results.sort_values(
        by=['title_match_score', 'rank_score', 'match_score', 'year'],
        ascending=[False, False, False, False]
    ).head(top_k).drop(columns=['title_match_score']).reset_index(drop=True)


# Test nhanh Boolean Search sau khi sửa ranking
print("AND — phải có cả transformer và attention:")
print(search_boolean("transformer attention", mode='AND')[['title', 'year', 'match_score', 'rank_score']].to_string())

print("\nNOT — có transformer nhưng không có recurrent:")
print(search_boolean("transformer recurrent", mode='NOT')[['title', 'year', 'match_score', 'rank_score']].to_string())


AND — phải có cả transformer và attention:
                                                                                                                     title  year  match_score  rank_score
0                                                 Attention is Not Only a Weight: Analyzing Transformers with Vector Norms  2020            2     10.6078
1                                                        Tree Transformer: Integrating Tree Structures into Self-Attention  2019            2     10.2959
2                                                                         Doubly Attentive Transformer Machine Translation  2018            2     10.2858
3                      Transformer Dissection: A Unified Understanding of Transformer's   Attention via the Lens of Kernel  2019            2     10.1584
4  Input-independent Attention Weights Are Expressive Enough: A Study of   Attention in Self-supervised Audio Transformers  2020            2     10.0365
5                                

In [7]:
import pickle, os

os.makedirs('/kaggle/working/models', exist_ok=True)

# Lưu TF-IDF
pickle.dump(tfidf_vectorizer,
            open('/kaggle/working/models/tfidf_vectorizer.pkl', 'wb'))
pickle.dump(tfidf_matrix,
            open('/kaggle/working/models/tfidf_matrix.pkl', 'wb'))

# Lưu BM25
pickle.dump(bm25,
            open('/kaggle/working/models/bm25.pkl', 'wb'))

# Lưu corpus tokens (cần khi load lại BM25)
pickle.dump(corpus_tokens,
            open('/kaggle/working/models/corpus_tokens.pkl', 'wb'))

print("Đã lưu xong! Files trong /kaggle/working/models/:")
for f in os.listdir('/kaggle/working/models'):
    size = os.path.getsize(f'/kaggle/working/models/{f}') / 1024 / 1024
    print(f"  {f:<35} {size:.1f} MB")

Đã lưu xong! Files trong /kaggle/working/models/:
  tfidf_matrix.pkl                    133.7 MB
  tfidf_vectorizer.pkl                4.0 MB
  corpus_tokens.pkl                   90.4 MB
  bm25.pkl                            74.0 MB


## Đánh giá bổ sung

Để so sánh TF-IDF và BM25 công bằng hơn, ta dùng chính title của một số paper làm query và coi các paper có trùng ít nhất một category là relevant. Đây là **pseudo-ground-truth**, chưa hoàn hảo nhưng đủ để báo cáo định lượng bước đầu.


In [8]:
from math import log2


cat_sets = df['categories'].fillna('').apply(lambda x: set(str(x).split())).to_list()
id_to_pos = pd.Series(df.index, index=df['id']).to_dict()


def _precision_at_k(results: pd.DataFrame, query_cats: set, k: int) -> float:
    if results.empty:
        return 0.0
    hits = sum(
        len(query_cats & cat_sets[id_to_pos[row['id']]]) > 0
        for _, row in results.head(k).iterrows()
        if row['id'] in id_to_pos
    )
    return hits / k


def _ndcg_at_k(results: pd.DataFrame, query_cats: set, k: int) -> float:
    rels = [
        1 if len(query_cats & cat_sets[id_to_pos[row['id']]]) > 0 else 0
        for _, row in results.head(k).iterrows()
        if row['id'] in id_to_pos
    ]
    dcg = sum(rel / log2(rank + 2) for rank, rel in enumerate(rels))
    idcg = sum(1 / log2(rank + 2) for rank in range(k))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_keyword_methods(test_size: int = 100, k_values=(5, 10), random_state: int = 42):
    rng = np.random.default_rng(random_state)
    sample_idx = rng.choice(len(df), size=min(test_size, len(df)), replace=False)
    records = []

    for idx in tqdm(sample_idx, desc='Evaluating keyword retrieval'):
        query = str(df.loc[idx, 'title'])
        query_id = df.loc[idx, 'id']
        query_cats = cat_sets[idx]

        for method_name, search_fn in [('TF-IDF', search_tfidf), ('BM25', search_bm25)]:
            max_k = max(k_values) + 1
            results = search_fn(query, top_k=max_k)
            results = results[results['id'] != query_id].head(max(k_values))

            for k in k_values:
                records.append({
                    'method': method_name,
                    'k': k,
                    'P@K': _precision_at_k(results, query_cats, k),
                    'NDCG@K': _ndcg_at_k(results, query_cats, k),
                })

    eval_df = pd.DataFrame(records)
    summary = eval_df.groupby(['method', 'k'])[['P@K', 'NDCG@K']].mean().round(4)
    print(summary)
    return eval_df, summary


# Chạy khi cần báo cáo định lượng:
keyword_eval_df, keyword_summary = evaluate_keyword_methods(test_size=100, k_values=(5, 10))


Evaluating keyword retrieval:   0%|          | 0/100 [00:00<?, ?it/s]

             P@K  NDCG@K
method k                
BM25   5   0.854  0.8553
       10  0.835  0.8419
TF-IDF 5   0.836  0.8442
       10  0.827  0.8356


## Nhận xét để đưa vào báo cáo

Keyword retrieval là baseline dễ giải thích, tốc độ tốt và phù hợp với truy vấn có thuật ngữ rõ ràng. BM25 thường nên được chọn làm baseline chính vì có cơ chế cân bằng tần suất từ và độ dài document. Hạn chế của hướng này là phụ thuộc vào từ khóa bề mặt, nên dễ kém hiệu quả khi người dùng diễn đạt bằng câu tự nhiên hoặc dùng từ đồng nghĩa.
